# 🎵 Music Genre Classification 
### DL & GenAI Project | Jan-2026 Term | IIT Madras

**Student:** 24f2008471  
**Project:** Messy Mashup Music Genre Classification (10 genres)  
**Metric:** Macro F1 Score  

---

## 📋 Table of Contents
1. [Setup & Imports](#setup)
2. [Model 1: SimpleCNN (Built from Scratch)](#cnn)
3. [Model 2: ResNet-18 (Pretrained Transfer Learning)](#resnet)
4. [Model 3: AST Transformer (Fine-tuned)](#ast)
5. [W&B Comparison & Results](#results)


---
## ⚙️ Section 1: Setup & Common Imports <a name="setup"></a>

Install all libraries needed by all 3 models.


In [1]:
!pip install -q transformers librosa torchaudio scikit-learn wandb torchvision tqdm

import os, random, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torchvision.models as torchvision_models
import torchvision.transforms as tv_transforms
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import (ASTFeatureExtractor, ASTForAudioClassification,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm
import wandb

warnings.filterwarnings('ignore')

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Common Paths ─────────────────────────────────────────────────────────────
STEMS_DIR   = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
NOISE_DIR   = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master'
TEST_CSV    = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
MASHUPS_DIR = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/mashups'

# ── W&B Login (one time) ─────────────────────────────────────────────────────
WANDB_KEY = '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'
wandb.login(key=WANDB_KEY)

# ── Genre Labels ──────────────────────────────────────────────────────────────
GENRES   = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
STEMS    = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
id2label = {i: g for i, g in enumerate(GENRES)}
label2id = {g: i for i, g in enumerate(GENRES)}

print('Setup complete!')


Using device: cpu


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2008471 (24f2008471-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setup complete!


---
## 🧠 Section 2: Model 1 — SimpleCNN (Built from Scratch) <a name="cnn"></a>

**Architecture:** 3 Convolutional blocks → Global Average Pooling → 2 Linear layers  
**Input:** Raw audio waveform → Mel Spectrogram (64 mel bins)  
**Why CNN?** A Mel Spectrogram is a 2D image (time × frequency). CNNs excel at detecting 2D patterns.

```
Audio Wave → MelSpectrogram (64×T) → Conv Block 1 (16 filters)
           → Conv Block 2 (32 filters) → Conv Block 3 (64 filters)
           → AdaptiveAvgPool → Linear(64→128) → Linear(128→10)
```


In [ ]:
# ── 2.1  Mel Spectrogram Transform ─────────────────────────────────────────

audio_to_mel = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=16000,  
        n_fft=1024,         
        hop_length=512,     
        n_mels=64           
    ),
    T.AmplitudeToDB()       
).to(device)

print('Mel Spectrogram transform ')


In [ ]:
# ── 2.2  Dataset — Mix 4 stems into one clip on the fly ────────────────────
class SimpleCNNDataset(Dataset):
    def __init__(self, stems_dir, config, mode='train'):
        self.sr         = config['sample_rate']
        self.target_len = int(self.sr * config['max_audio_length_sec'])

        self.genre_songs = {g: [] for g in GENRES}
        for g in GENRES:
            g_path = os.path.join(stems_dir, g)
            if os.path.exists(g_path):
                self.genre_songs[g] = [
                    os.path.join(g_path, d)
                    for d in os.listdir(g_path)
                    if os.path.isdir(os.path.join(g_path, d))
                ]

        self.epoch_size = 2000 if mode == 'train' else 400

    def __len__(self):
        return self.epoch_size

    def _load_and_pad(self, path):
        wave, orig_sr = torchaudio.load(path)
        if orig_sr != self.sr:
            wave = torchaudio.functional.resample(wave, orig_sr, self.sr)
        wave = wave[0]  # stereo → mono

        if wave.shape[0] >= self.target_len:
            start = random.randint(0, wave.shape[0] - self.target_len)
            wave  = wave[start:start + self.target_len]
        else:
            wave  = torch.nn.functional.pad(wave, (0, self.target_len - wave.shape[0]))
        return wave

    def __getitem__(self, idx):
        label_idx    = random.randint(0, len(GENRES) - 1)
        target_genre = GENRES[label_idx]

        # Mix the 4 stems (bass + drums + other + vocals)
        mixed = torch.zeros(self.target_len)
        for stem in STEMS:
            song_path = random.choice(self.genre_songs[target_genre])
            stem_path = os.path.join(song_path, stem)
            if os.path.exists(stem_path):
                wave   = self._load_and_pad(stem_path)
                mixed += wave * random.uniform(0.7, 1.3)  # random volume

        # Normalize to [-1, 1]
        max_val = torch.max(torch.abs(mixed))
        if max_val > 0:
            mixed = mixed / max_val

        return mixed, torch.tensor(label_idx, dtype=torch.long)

print('SimpleCNNDataset ')


In [ ]:
# ── 2.3  Model Architecture ────────────────────────────────────────────────
class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),           
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),         
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn_model  = SimpleCNN(num_classes=10).to(device)
total_p = sum(p.numel() for p in cnn_model.parameters())
print(f'SimpleCNN — Total parameters: {total_p:,}')


In [ ]:
# ── 2.4  Train SimpleCNN ───────────────────────────────────────────────────
cnn_config = {
    'epochs': 20, 'batch_size': 32, 'learning_rate': 1e-3,
    'sample_rate': 16000, 'max_audio_length_sec': 5.0,
    'n_mels': 64, 'architecture': 'SimpleCNN-3Block-Scratch'
}

run_cnn = wandb.init(
    entity='24f2008471-indian-institute-of-technology-madras',
    project='Dl-genai-26-t1',
    name='Model_SimpleCNN_Scratch',
    config=cnn_config,
    reinit=True
)

train_dataset_cnn = SimpleCNNDataset(STEMS_DIR, wandb.config, mode='train')
train_loader_cnn  = DataLoader(train_dataset_cnn, batch_size=wandb.config.batch_size,
                               shuffle=True, num_workers=0)

criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = torch.optim.Adam(cnn_model.parameters(), lr=wandb.config.learning_rate)

print(f'Batches per epoch: {len(train_loader_cnn)}')
print('Starting SimpleCNN training...')

for epoch in range(wandb.config.epochs):
    cnn_model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for waves, labels in tqdm(train_loader_cnn, desc=f'CNN Epoch {epoch+1}/{wandb.config.epochs}'):
        waves, labels = waves.to(device), labels.to(device)

        with torch.no_grad():
            mel = audio_to_mel(waves).unsqueeze(1)   # (B,1,64,T)

        optimizer_cnn.zero_grad()
        logits = cnn_model(mel)
        loss   = criterion_cnn(logits, labels)
        loss.backward()
        optimizer_cnn.step()

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader_cnn)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    wandb.log({'epoch': epoch+1, 'train_loss': avg_loss,
               'train_accuracy': acc, 'train_macro_f1': f1})
    print(f'Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')
    torch.save(cnn_model.state_dict(), f'simple_cnn_epoch_{epoch+1}.pt')

wandb.finish()
print('SimpleCNN training complete!')


In [ ]:
# ── 2.5  SimpleCNN Inference ──────────────────────────────────────────────
class CNNTestDataset(Dataset):
    
    def __init__(self, test_csv, audio_dir, sample_rate=16000):
        self.df     = pd.read_csv(test_csv, dtype={'id': str})
        self.sr     = sample_rate
        self.window = self.sr * 5    # 80,000 samples
        self.step   = self.sr * 2    # 60% overlap

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(audio_dir, os.path.basename(row['filename']))
        try:
            wave, orig_sr = torchaudio.load(path)
            if orig_sr != self.sr:
                wave = torchaudio.functional.resample(wave, orig_sr, self.sr)
            y = wave[0]
        except:
            y = torch.zeros(self.window)

        chunks, n = [], len(y)
        if n < self.window:
            chunks.append(torch.nn.functional.pad(y, (0, self.window - n)))
        else:
            for s in range(0, n - self.window + 1, self.step):
                chunks.append(y[s:s + self.window])
            chunks.append(y[-self.window:])

        return torch.stack(chunks), row['id']

cnn_model.load_state_dict(torch.load('simple_cnn_epoch_20.pt'))
cnn_model.eval()

test_loader_cnn = DataLoader(CNNTestDataset(TEST_CSV, MASHUPS_DIR),
                             batch_size=1, shuffle=False, num_workers=0)
all_ids, all_preds = [], []
with torch.no_grad():
    for chunks, file_id in tqdm(test_loader_cnn, desc='CNN Inference'):
        chunks = chunks.squeeze(0).to(device)
        mel    = audio_to_mel(chunks).unsqueeze(1)
        logits = cnn_model(mel)
        pred   = torch.argmax(logits.mean(dim=0)).item()
        all_ids.append(file_id[0])
        all_preds.append(pred)

pd.DataFrame({'id': [str(i).zfill(4) for i in all_ids],
              'genre': [id2label[p] for p in all_preds]}).to_csv(
    'submission_simple_cnn.csv', index=False)
print('Saved: submission_simple_cnn.csv')


---
##  Section 3: Model 2 — ResNet-18 (Pretrained Transfer Learning) <a name="resnet"></a>

**Architecture:** ResNet-18 pretrained on ImageNet, head replaced for 10-class audio classification  
**Input:** Audio → 128×128 Mel Spectrogram (3-channel, ImageNet-normalized)  
**Why Transfer Learning?** ResNet-18 already knows how to detect visual patterns. We re-use its feature extraction and fine-tune only the top layers.  
**Key Differences from SimpleCNN:**
- Residual connections prevent vanishing gradients
- 11M parameters vs SimpleCNN's ~50K
- Layers 1 & 2 frozen; layers 3 & 4 + head trainable


In [ ]:
# ── 3.1  Audio → 128×128 3-channel Mel Spectrogram ────────────────────────
RESNET_SR         = 22050
RESNET_N_MELS     = 128
RESNET_HOP_LENGTH = 512
RESNET_N_FFT      = 2048
RESNET_IMG_SIZE   = 128

def audio_to_melspec_resnet(y, sr=RESNET_SR):
    mel    = librosa.feature.melspectrogram(y=y, sr=sr,
                n_mels=RESNET_N_MELS, hop_length=RESNET_HOP_LENGTH, n_fft=RESNET_N_FFT)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    t = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    t = torch.nn.functional.interpolate(t, size=(RESNET_IMG_SIZE, RESNET_IMG_SIZE),
                                         mode='bilinear', align_corners=False).squeeze(0)
    return t.repeat(3, 1, 1)  

IMAGENET_NORM = tv_transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                         std =[0.229, 0.224, 0.225])


In [ ]:
# ── 3.2  Dataset ──────────────────────────────────────────────────────────
class ResNetDataset(Dataset):
    def __init__(self, stems_dir, noise_dir, augment=True, epoch_size=1500):
        self.augment    = augment
        self.epoch_size = epoch_size

        self.noise_files = glob.glob(os.path.join(noise_dir, 'audio', '*.wav'))

        self.genre_songs = {g: [] for g in GENRES}
        for g in GENRES:
            g_path = os.path.join(stems_dir, g)
            if os.path.exists(g_path):
                self.genre_songs[g] = [
                    os.path.join(g_path, d)
                    for d in os.listdir(g_path)
                    if os.path.isdir(os.path.join(g_path, d))
                ]

    def __len__(self): return self.epoch_size

    def _make_mashup(self, genre):
        target_len = int(RESNET_SR * 5.0)
        mixed      = np.zeros(target_len, dtype=np.float32)

        # Cross-song mixing: different song folder per stem
        song_paths = random.choices(self.genre_songs[genre], k=len(STEMS))
        for stem, song_path in zip(STEMS, song_paths):
            stem_path = os.path.join(song_path, stem)
            if not os.path.exists(stem_path): continue
            try:
                wave, _ = librosa.load(stem_path, sr=RESNET_SR, mono=True)
            except: continue
            if len(wave) >= target_len:
                start = random.randint(0, len(wave) - target_len)
                wave  = wave[start:start + target_len]
            else:
                wave = np.tile(wave, int(np.ceil(target_len / len(wave))))[:target_len]
            mixed += wave * random.uniform(0.7, 1.3)

        # Add ESC-50 background noise (80% of the time)
        if random.random() < 0.8 and self.noise_files:
            try:
                noise, _ = librosa.load(random.choice(self.noise_files),
                                        sr=RESNET_SR, mono=True)
                if len(noise) >= target_len: noise = noise[:target_len]
                else: noise = np.tile(noise, int(np.ceil(target_len/len(noise))))[:target_len]
                mixed += noise * random.uniform(0.05, 0.3)
            except: pass

        mx = np.abs(mixed).max()
        if mx > 0: mixed /= mx
        return mixed

    def __getitem__(self, idx):
        label_idx = random.randint(0, len(GENRES) - 1)
        wave = self._make_mashup(GENRES[label_idx])
        if self.augment and random.random() < 0.5:
            wave = wave * random.uniform(0.8, 1.2)
        mel = IMAGENET_NORM(audio_to_melspec_resnet(wave))
        return mel, torch.tensor(label_idx, dtype=torch.long)



In [ ]:
# ── 3.3  Build ResNet-18 ──────────────────────────────────────────────────
def build_resnet18(num_classes=10, dropout=0.3):
    model = torchvision_models.resnet18(weights=torchvision_models.ResNet18_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        if 'layer1' in name or 'layer2' in name:
            param.requires_grad = False       # Freeze early layers
    model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(512, num_classes))
    return model

resnet_model = build_resnet18().to(device)
total   = sum(p.numel() for p in resnet_model.parameters())
trained = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trained:,}')
print(f'Frozen params   : {total - trained:,}')


In [ ]:
# ── 3.4  Train ResNet-18 ──────────────────────────────────────────────────
resnet_config = {
    'model': 'ResNet-18', 'epochs': 12, 'batch_size': 64,
    'lr': 1e-3, 'lr_backbone': 1e-4, 'epoch_size': 1500,
    'dropout': 0.3, 'noise_prob': 0.8, 'cross_song_mix': True,
}

run_resnet = wandb.init(
    entity='24f2008471-indian-institute-of-technology-madras',
    project='Dl-genai-26-t1',
    name='ResNet18_Fast_v1',
    config=resnet_config,
    reinit=True
)

train_dataset_rn = ResNetDataset(STEMS_DIR, NOISE_DIR, augment=True,
                                  epoch_size=wandb.config.epoch_size)
train_loader_rn  = DataLoader(train_dataset_rn, batch_size=wandb.config.batch_size,
                               shuffle=True, num_workers=0, pin_memory=True)

# Two learning-rate groups: lower LR for backbone, higher for new head
backbone_params = [p for n, p in resnet_model.named_parameters()
                   if p.requires_grad and 'fc' not in n]
head_params     = list(resnet_model.fc.parameters())

optimizer_rn = torch.optim.AdamW([
    {'params': backbone_params, 'lr': wandb.config.lr_backbone},
    {'params': head_params,     'lr': wandb.config.lr},
], weight_decay=1e-4)

scheduler_rn = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_rn, T_max=wandb.config.epochs, eta_min=1e-6)

criterion_rn = nn.CrossEntropyLoss(label_smoothing=0.1)

best_f1_rn, best_epoch_rn = 0.0, 0

print('Starting ResNet-18 training...')
for epoch in range(wandb.config.epochs):
    resnet_model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for images, labels in tqdm(train_loader_rn, desc=f'ResNet Epoch {epoch+1}/{wandb.config.epochs}'):
        images, labels = images.to(device), labels.to(device)
        optimizer_rn.zero_grad()
        outputs = resnet_model(images)
        loss = criterion_rn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(resnet_model.parameters(), 1.0)
        optimizer_rn.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    scheduler_rn.step()
    avg_loss = total_loss / len(train_loader_rn)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    print(f'Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')
    wandb.log({'epoch': epoch+1, 'train_loss': avg_loss,
               'train_accuracy': acc, 'train_macro_f1': f1})

    if f1 > best_f1_rn:
        best_f1_rn, best_epoch_rn = f1, epoch+1
        torch.save(resnet_model.state_dict(), 'resnet18_best.pt')

wandb.run.summary.update({'best_f1': best_f1_rn, 'best_epoch': best_epoch_rn})
wandb.finish()
print(f'ResNet-18 done. Best F1: {best_f1_rn:.4f} at epoch {best_epoch_rn}')


In [ ]:
# ── 3.5  ResNet-18 Inference ──────────────────────────────────────────────
class ResNetTestDataset(Dataset):
    def __init__(self, test_csv, audio_dir, window_sec=5.0, step_sec=1.5):
        self.df     = pd.read_csv(test_csv, dtype={'id': str})
        self.audio_dir = audio_dir
        self.window = int(RESNET_SR * window_sec)
        self.step   = int(RESNET_SR * step_sec)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, os.path.basename(row['filename']))
        try:
            y, _ = librosa.load(path, sr=RESNET_SR, mono=True)
        except:
            y = np.zeros(self.window)

        chunks, n = [], len(y)
        if n < self.window:
            chunks.append(np.pad(y, (0, self.window - n)))
        else:
            for s in range(0, n - self.window + 1, self.step):
                chunks.append(y[s:s + self.window])
            if (n - self.window) % self.step != 0:
                chunks.append(y[-self.window:])

        mels = [IMAGENET_NORM(audio_to_melspec_resnet(c)) for c in chunks]
        return torch.stack(mels), row['id']

resnet_model.load_state_dict(torch.load('resnet18_best.pt', map_location=device))
resnet_model.eval()

test_loader_rn = DataLoader(ResNetTestDataset(TEST_CSV, MASHUPS_DIR),
                             batch_size=1, shuffle=False, num_workers=0)
all_ids, all_preds = [], []
with torch.no_grad():
    for mels, file_id in tqdm(test_loader_rn, desc='ResNet Inference'):
        mels    = mels.squeeze(0).to(device)
        probs   = torch.softmax(resnet_model(mels), dim=-1).mean(dim=0)
        all_ids.extend(file_id)
        all_preds.append(torch.argmax(probs).item())

pd.DataFrame({'id': [str(i).zfill(4) for i in all_ids],
              'genre': [id2label[p] for p in all_preds]}).to_csv(
    'submission_resnet18.csv', index=False)
print('Saved: submission_resnet18.csv')


---
## 🤖 Section 4: Model 3 — AST Transformer (Fine-Tuned Pretrained) <a name="ast"></a>

**Architecture:** Audio Spectrogram Transformer (AST) — pretrained on AudioSet (2M clips, 527 classes)  
**Input:** Raw audio waveform → processed by ASTFeatureExtractor internally  
**Why AST?** Transformers use self-attention to capture long-range dependencies across time — important for musical structure (verse/chorus/bridge patterns).  

**Key Training Techniques:**
- **SpecAugment:** Randomly mask frequency & time bands to prevent overfitting
- **Cosine LR with Warmup:** Gently ramp up LR to avoid destroying pretrained weights
- **Noise Injection:** Add ESC-50 environmental noise to simulate the test mashups


In [ ]:
# ── 4.1  W&B Config & AST Model Setup ────────────────────────────────────
ast_config = {
    'epochs': 10, 'batch_size': 8, 'learning_rate': 5e-5,
    'max_audio_length_sec': 5.0, 'noise_injection_prob': 0.5,
    'architecture': 'AST-Advanced', 'scheduler': 'Cosine with Warmup',
    'augmentations': 'SpecAugment',
}

run_ast = wandb.init(
    entity='24f2008471-indian-institute-of-technology-madras',
    project='Dl-genai-26-t1',
    name='AST_Kaggle_Run_3_Pro_Pipeline',
    config=ast_config,
    reinit=True
)

feature_extractor = ASTFeatureExtractor.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593')
AST_SR = feature_extractor.sampling_rate  # 16000

# Load pretrained AST, replace 527-class head with 10-class head
ast_model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=10,
    label2id={g: str(i) for i, g in enumerate(GENRES)},
    id2label={str(i): g for i, g in enumerate(GENRES)},
    ignore_mismatched_sizes=True   # new head replaces old 527-class head
).to(device)

print('AST model loaded.')


In [ ]:
# ── 4.2  Dataset ─────────────────────────────────────────────────────────
class ASTDataset(Dataset):
    def __init__(self, stems_dir, noise_dir, feature_extractor, config, mode='train'):
        self.feature_extractor = feature_extractor
        self.config = config
        self.sr     = feature_extractor.sampling_rate
        self.noise_files = glob.glob(os.path.join(noise_dir, 'audio', '*.wav'))

        self.genre_songs = {g: [] for g in GENRES}
        for g in GENRES:
            g_path = os.path.join(stems_dir, g)
            if os.path.exists(g_path):
                self.genre_songs[g] = [
                    os.path.join(g_path, d)
                    for d in os.listdir(g_path)
                    if os.path.isdir(os.path.join(g_path, d))
                ]
        self.epoch_size = 2000 if mode == 'train' else 500

    def __len__(self): return self.epoch_size

    def __getitem__(self, idx):
        label_idx    = random.randint(0, len(GENRES) - 1)
        target_genre = GENRES[label_idx]
        target_len   = int(self.sr * self.config['max_audio_length_sec'])
        mixed        = np.zeros(target_len)

        # Tempo-synced stem mixing
        base_stretch = random.uniform(0.85, 1.15)
        for stem in STEMS:
            song_path = random.choice(self.genre_songs[target_genre])
            stem_path = os.path.join(song_path, stem)
            if not os.path.exists(stem_path): continue
            wave, _ = librosa.load(stem_path, sr=self.sr, mono=True)
            stretch  = base_stretch * random.uniform(0.95, 1.05)
            wave     = librosa.effects.time_stretch(wave, rate=stretch)
            if len(wave) > target_len: wave = wave[:target_len]
            else: wave = np.pad(wave, (0, target_len - len(wave)))
            mixed += wave * random.uniform(0.7, 1.3)

        # ESC-50 noise injection
        if random.random() < self.config['noise_injection_prob'] and self.noise_files:
            noise, _ = librosa.load(random.choice(self.noise_files), sr=self.sr, mono=True)
            if len(noise) > target_len: noise = noise[:target_len]
            else: noise = np.pad(noise, (0, target_len - len(noise)))
            mixed += noise * random.uniform(0.1, 0.4)

        mx = np.abs(mixed).max()
        if mx > 0: mixed /= mx

        inputs = self.feature_extractor(mixed, sampling_rate=self.sr, return_tensors='pt')
        return inputs['input_values'].squeeze(0), torch.tensor(label_idx, dtype=torch.long)

print('ASTDataset defined.')


In [ ]:
# ── 4.3  DataLoader + Optimizer + Scheduler ──────────────────────────────
train_dataset_ast = ASTDataset(STEMS_DIR, NOISE_DIR, feature_extractor,
                               wandb.config, mode='train')
train_loader_ast  = DataLoader(train_dataset_ast, batch_size=wandb.config.batch_size,
                               shuffle=True, num_workers=2)

optimizer_ast = torch.optim.AdamW(ast_model.parameters(),
                                  lr=wandb.config.learning_rate, weight_decay=0.01)

total_steps  = len(train_loader_ast) * wandb.config.epochs
warmup_steps = int(0.1 * total_steps)   # 10% warmup

scheduler_ast = get_cosine_schedule_with_warmup(
    optimizer_ast,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# SpecAugment: randomly mask frequency / time bands
freq_masker = T.FrequencyMasking(freq_mask_param=15).to(device)
time_masker = T.TimeMasking(time_mask_param=35).to(device)

print(f'Total training steps: {total_steps}  |  Warmup steps: {warmup_steps}')


In [ ]:
# ── 4.4  Train AST ───────────────────────────────────────────────────────
print('Starting AST fine-tuning...')

for epoch in range(wandb.config.epochs):
    ast_model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for inputs, labels in tqdm(train_loader_ast,
                               desc=f'AST Epoch {epoch+1}/{wandb.config.epochs}'):
        inputs, labels = inputs.to(device), labels.to(device)

        # SpecAugment: mask frequency/time bands for regularisation
        inputs_t  = inputs.transpose(1, 2)
        inputs_t  = freq_masker(inputs_t)
        inputs_t  = time_masker(inputs_t)
        inputs    = inputs_t.transpose(1, 2)

        optimizer_ast.zero_grad()
        outputs = ast_model(inputs, labels=labels)
        loss    = outputs.loss
        loss.backward()
        optimizer_ast.step()
        scheduler_ast.step()   # step every batch (not every epoch)

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        wandb.log({'batch_loss': loss.item(), 'lr': scheduler_ast.get_last_lr()[0]})

    avg_loss = total_loss / len(train_loader_ast)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    print(f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')
    wandb.log({'epoch': epoch+1, 'train_loss': avg_loss,
               'train_accuracy': acc, 'train_macro_f1': f1})
    torch.save(ast_model.state_dict(), f'ast_model_epoch_{epoch+1}.pt')

wandb.finish()
print('AST fine-tuning complete!')


In [ ]:
# ── 4.5  AST Inference (Fast Sliding Window) ─────────────────────────────
AST_BEST_MODEL_PATH = '/kaggle/input/datasets/lucydone/best-epoch/ast_model_epoch_10.pt'

class ASTTestDataset(Dataset):
    def __init__(self, test_csv, audio_dir, feature_extractor,
                 window_sec=5.0, step_sec=1.5):
        self.df = pd.read_csv(test_csv, dtype={'id': str})
        self.audio_dir = audio_dir
        self.feature_extractor = feature_extractor
        self.sr     = feature_extractor.sampling_rate
        self.window = int(self.sr * window_sec)
        self.step   = int(self.sr * step_sec)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.audio_dir, os.path.basename(row['filename']))
        try:
            y, _ = librosa.load(path, sr=self.sr, mono=True)
        except:
            y = np.zeros(self.window)

        chunks, n = [], len(y)
        if n < self.window:
            chunks.append(np.pad(y, (0, self.window - n)))
        else:
            for s in range(0, n - self.window + 1, self.step):
                chunks.append(y[s:s + self.window])
            if (n - self.window) % self.step != 0:
                chunks.append(y[-self.window:])

        inputs = self.feature_extractor(chunks, sampling_rate=self.sr, return_tensors='pt')
        return inputs['input_values'], row['id']

ast_model.load_state_dict(torch.load(AST_BEST_MODEL_PATH, map_location=device))
ast_model.eval()

test_loader_ast = DataLoader(ASTTestDataset(TEST_CSV, MASHUPS_DIR, feature_extractor),
                              batch_size=1, shuffle=False, num_workers=0)
all_ids, all_preds = [], []
with torch.no_grad():
    for inputs, file_id in tqdm(test_loader_ast, desc='AST Inference'):
        inputs      = inputs.squeeze(0).to(device)
        mean_logits = ast_model(inputs).logits.mean(dim=0)
        all_ids.extend(file_id)
        all_preds.append(torch.argmax(mean_logits).item())

pd.DataFrame({'id': [str(i).zfill(4) for i in all_ids],
              'genre': [id2label[p] for p in all_preds]}).to_csv(
    'submission_fast.csv', index=False)
print('Saved: submission_fast.csv')


---
## 📊 Section 5: W&B Model Comparison & Results Summary <a name="results"></a>

All 3 model runs are logged to the same W&B project: **`Dl-genai-26-t1`**  
View & compare at: [wandb.ai/24f2008471.../Dl-genai-26-t1](https://wandb.ai/24f2008471-indian-institute-of-technology-madras/Dl-genai-26-t1)

---

### 🏆 Kaggle Leaderboard Scores

| Model | Type | Kaggle Score  | W&B Run Name |
|---|---|---|---|---|
| SimpleCNN | From Scratch | 0.45263 |  `Model_SimpleCNN_Scratch` |
| ResNet-18 | Transfer Learning | 0.82880 | `ResNet18_Fast_v1` |
| **AST Transformer** | **Fine-Tuned Pretrained** | **0.92064** | `AST_Kaggle_Run_3_Pro_Pipeline` |

---

### 📈 Training Progression (Final Epoch Metrics)

| Model | Epochs | Final Train Loss | Final Train Acc | Final Train F1 |
|---|---|---|---|---|
| SimpleCNN | 20 | 1.1133 | 0.6095 | 0.6033 |
| ResNet-18 | 12 | 1.0427 | 0.7733 | 0.7743 |
| AST Transformer | 10 | 0.1849 | 0.9350 | **0.9351** |

---

### 🔍 Key Observations & Analysis

**SimpleCNN (0.45263):**
- Started at F1=0.16 (epoch 1) → improved to F1=0.60 by epoch 20 — steady learning
- Only 32,906 parameters — limited capacity to capture complex audio patterns
- Train F1 (0.60) >> Kaggle score (0.45) → overfitting to training distribution
- Useful as a baseline to show that even a tiny CNN can learn genre structure

**ResNet-18 (0.82880):**
- Crossed 0.80 cutoff — proves transfer learning works for audio
- Cross-song stem mixing + ESC-50 noise augmentation helped generalise to test mashups
- 100 songs × 10 genres = 1000 training songs; ResNet leveraged ImageNet visual features
- Frozen layers 1 & 2 preserved low-level feature detectors; layers 3 & 4 adapted to audio

**AST Transformer (0.92064):**
- Loss dropped from 1.30 → 0.18 in just 10 epochs — very fast convergence
- Self-attention mechanism captures long-range dependencies (e.g., verse-chorus patterns)
- SpecAugment + cosine warmup scheduler prevented overfitting despite smaller dataset
- Pretrained on AudioSet (2M audio clips, 527 classes) — massive prior knowledge

---

### 📉 Why does train F1 not equal Kaggle score?
The training data uses **individual 5-second stem clips**, while test data contains **full-length mashups** of songs from multiple genres mixed together. The domain gap explains why even AST's train F1 (0.9351) > Kaggle score (0.9206).


In [ ]:
# ── 5.1  Print final submission distribution ──────────────────────────────
for fname, model_name in [('submission_simple_cnn.csv', 'SimpleCNN'),
                           ('submission_resnet18.csv',   'ResNet-18'),
                           ('submission_fast.csv',       'AST')]:
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        print(f'\n{model_name} predictions ({len(df)} samples):')
        print(df['genre'].value_counts().sort_index().to_string())
